In [ ]:
import pandas as pd
import numpy as np

## Paths

In [ ]:
data_path = "../data/processed"

In [ ]:
from eda_toolkit import ensure_directory
import os  # import operating system for dir

base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_pdf = os.path.join(base_path, "images", "pdf_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_pdf)
ensure_directory(image_path_svg)

In [ ]:
import json
from pathlib import Path

pred_dir = Path("../models/predictions/full_text_clean")

X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
y

In [ ]:
print("X index:", X.index.name, X.index[:3].tolist(), "n =", len(X))
print("df_time index:", df_time.index.name, df_time.index[:3].tolist(), "n =", len(df_time))
print("df_time columns:", df_time.columns.tolist())

In [ ]:
"""
Cox proportional hazards sensitivity analysis.

The primary analysis dichotomizes the outcome and excludes follow-up duration
from the feature set, because duration is mechanically determined by the
outcome and would leak it. That leaves an obvious question: does collapsing
the time-to-event structure distort the associations? This fits a Cox model on
the same nine features, using duration as the time variable rather than as a
predictor, so the two framings can be compared.

Reported as a sensitivity analysis only. Not a competing model, and not for
the abstract.

The SHAP comparison reads `shap_importance.parquet` if present, rather than
depending on `importance` surviving in kernel state across notebooks. Write it
from the SHAP notebook with:

    importance.to_parquet("../data/processed/shap_importance.parquet")
"""

from pathlib import Path

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

DURATION = "time_months"
EVENT = "outcome"
PROCESSED = Path("../data/processed")
SHAP_PATH = PROCESSED / "shap_importance.parquet"

# --- assemble: features from X, follow-up time from the preprocessed file --
df_time = pd.read_parquet(PROCESSED / "df_sans_zero.parquet").set_index("id")

cox_df = X.copy()

# preprocessing artifact, constant at zero here; would either produce a
# meaningless coefficient or block convergence
cox_df = cox_df.drop(columns=["percentage_missing"], errors="ignore")

cox_df[EVENT] = y["outcome"].values
cox_df[DURATION] = df_time.loc[X.index, DURATION].to_numpy()

# joining on id rather than position; confirm the rows really do line up
assert np.allclose(
    df_time.loc[X.index, "creatinine"].to_numpy(), X["creatinine"].to_numpy()
), "row alignment mismatch between X and df_sans_zero"

# Two patients have a recorded duration of zero, both deaths. Cox cannot use a
# zero duration. Half a month retains them; with a median follow-up near nine
# years the substitution has no material effect.
n_zero = int((cox_df[DURATION] <= 0).sum())
if n_zero:
    print(f"{n_zero} patient(s) with non-positive follow-up, clipped to 0.5 months")
cox_df[DURATION] = cox_df[DURATION].clip(lower=0.5)

assert cox_df[DURATION].notna().all(), "missing follow-up time"
assert (cox_df[DURATION] > 0).all(), "non-positive follow-up time"

print(f"n = {len(cox_df)}, events = {int(cox_df[EVENT].sum())}, "
      f"median follow-up = {cox_df[DURATION].median():.1f} months")
print(f"covariates: {[c for c in cox_df.columns if c not in (EVENT, DURATION)]}")

# --- fit -------------------------------------------------------------------
# No weights_col. Passing a covariate there makes lifelines treat it as
# frequency weights, silently inflating the sample and rendering the
# concordance uninterpretable.
cph = CoxPHFitter()
cph.fit(cox_df, duration_col=DURATION, event_col=EVENT)

assert abs(cph._n_examples - len(cox_df)) < 1, (
    f"lifelines saw {cph._n_examples} observations, expected {len(cox_df)}"
)

cph.print_summary(decimals=3)
print(f"\nconcordance = {cph.concordance_index_:.3f}")

# --- proportional hazards assumption ---------------------------------------
print("\nProportional hazards test:")
ph = proportional_hazard_test(cph, cox_df, time_transform="rank")
ph.print_summary(decimals=3)

ph_flags = ph.summary[ph.summary["p"] < 0.05].index.tolist()
print(f"\ncovariates violating proportional hazards at p < 0.05: "
      f"{ph_flags if ph_flags else 'none'}")

# --- Supplementary Table S2 ------------------------------------------------
summary = cph.summary[
    ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]
].copy()
summary.columns = ["HR", "HR lower 95%", "HR upper 95%", "p"]
summary.index.name = "Feature"

BACK = {
    "Serum creatinine": "creatinine",
    "Cardiovascular disease": "cardiovascular_disease",
    "Diabetes mellitus": "diabetes",
    "Dyslipidemia": "dyslipidemia",
    "Smoking": "smoking",
    "Cancer": "cancer",
    "Hypertension": "hypertension",
    "Sex": "sex",
    "Obesity": "obesity",
}

shap_rank = None
if SHAP_PATH.exists():
    imp = pd.read_parquet(SHAP_PATH)
    col = next((c for c in imp.columns if "SHAP" in c.upper()), None)
    if col is not None:
        shap_rank = imp.set_index("Feature")[col].rename("Mean |SHAP|")
        shap_rank.index = [BACK.get(i, i) for i in shap_rank.index]
    else:
        print(f"\nno SHAP column found in {SHAP_PATH.name}: {list(imp.columns)}")
else:
    print(f"\n{SHAP_PATH.name} not found; reporting hazard ratios only")

if shap_rank is not None:
    cox_table = summary.join(shap_rank, how="left").sort_values("HR", ascending=False)
    cox_table["SHAP rank"] = (
        cox_table["Mean |SHAP|"].rank(ascending=False).astype("Int64")
    )
    rho = cox_table["HR"].rank().corr(cox_table["Mean |SHAP|"].rank(),
                                      method="spearman")
    print(f"\nSpearman rank correlation, hazard ratio vs mean |SHAP|: {rho:.3f}")
else:
    cox_table = summary.sort_values("HR", ascending=False)

print("\nSupplementary Table S2, Cox proportional hazards model:")
print(cox_table.round(3).to_string())

cox_table.to_parquet(PROCESSED / "cox_table_s2.parquet")
print(f"\nsaved {PROCESSED / 'cox_table_s2.parquet'}")

In [ ]:
cph.check_assumptions(cox_df, p_value_threshold=0.05, show_plots=True)